In [ ]:
pip install torch

In [ ]:
!pip install pin
import pinocchio
print(dir(pinocchio))

['ACCELERATION', 'ADMMContactSolver', 'ARG0', 'ARG1', 'ARG2', 'ARG3', 'ARG4', 'AngleAxis', 'ArgumentPosition', 'BODY', 'BaumgarteCorrectorParameters', 'BroadPhaseManager_DynamicAABBTreeArrayCollisionManager', 'BroadPhaseManager_DynamicAABBTreeCollisionManager', 'BroadPhaseManager_IntervalTreeCollisionManager', 'BroadPhaseManager_NaiveCollisionManager', 'BroadPhaseManager_SSaPCollisionManager', 'BroadPhaseManager_SaPCollisionManager', 'COLLISION', 'CachedMeshLoader', 'CollisionCallBackBase', 'CollisionCallBackDefault', 'CollisionGeometry', 'CollisionObject', 'CollisionPair', 'CollisionResult', 'ComputeCollision', 'ComputeDistance', 'Contact', 'ContactCholeskyDecomposition', 'ContactType', 'Convention', 'CoulombFrictionCone', 'Data', 'DelassusCholeskyExpression', 'DelassusOperatorDense', 'DelassusOperatorSparse', 'DistanceResult', 'DualCoulombFrictionCone', 'Exception', 'FIXED_JOINT', 'Force', 'Frame', 'FrameType', 'GeometryData', 'GeometryModel', 'GeometryNoMaterial', 'GeometryObject', 

In [ ]:
pip install numpy scipy

In [ ]:
import pinocchio as pin
import torch
import time

# ----------------------------
# GPU + dtype setup
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64  # match numpy double precision
print("Using device:", device)

# ----------------------------
# Algorithm Implementations
# ----------------------------
def gauss_jordan(M, b):
    return torch.linalg.solve(M, b)

def neumann_series_inverse(M, num_terms=10):
    M0 = torch.diag(torch.diag(M))
    M0_inv = torch.linalg.inv(M0)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - M0_inv @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_terms):
        term = term @ E
        S = S + term
    return S @ M0_inv

def spai_inverse(M):
    return torch.diag(1.0 / torch.diag(M))

def hala(M, b, num_neumann=5):
    G_spai = spai_inverse(M)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - G_spai @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_neumann):
        term = term @ E
        S = S + term
    M_inv_hala = S @ G_spai
    qddot_hala = M_inv_hala @ b
    error = torch.linalg.norm(M @ qddot_hala - b)
    if error > 1e-3:
        qddot_hala = gauss_jordan(M, b)
    return qddot_hala

# ----------------------------
# Setup Pinocchio model
# ----------------------------
urdf_path = '/content/drive/MyDrive/ur5robot.urdf'
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()
nq, nv = model.nq, model.nv

q_lower, q_upper = model.lowerPositionLimit, model.upperPositionLimit
qd_limit = model.velocityLimit

# Ensure finite, reasonable velocity limits
qd_limit_safe = qd_limit.copy()
qd_limit_safe[~torch.isfinite(torch.tensor(qd_limit_safe))] = 1.0
qd_limit_safe = torch.clamp(torch.tensor(qd_limit_safe), 0, 10).numpy()

# ----------------------------
# Random Sampling
# ----------------------------
num_runs = 1000
q_lower_t = torch.tensor(q_lower, dtype=dtype)
q_upper_t = torch.tensor(q_upper, dtype=dtype)
qd_limit_safe_t = torch.tensor(qd_limit_safe, dtype=dtype)

# Uniformly sample between lower and upper joint limits
q_samples = q_lower_t + (q_upper_t - q_lower_t) * torch.rand(num_runs, nq, dtype=dtype)
qd_samples = -qd_limit_safe_t + 2 * qd_limit_safe_t * torch.rand(num_runs, nv, dtype=dtype)

# ----------------------------
# Benchmark Setup
# ----------------------------
results = {
    'ref_time': [], 'neumann_time': [], 'spai_time': [], 'hala_time': [],
    'neumann_error': [], 'spai_error': [], 'hala_error': [],
    'ref_kappa': [], 'neumann_kappa': [], 'spai_kappa': [], 'hala_kappa': []
}

# ----------------------------
# Benchmark Loop
# ----------------------------
for i in range(num_runs):
    q = q_samples[i].cpu().numpy()
    qd = qd_samples[i].cpu().numpy()
    pin.computeAllTerms(model, data, q, qd)

    # Convert Pinocchio outputs to GPU tensors
    M = torch.tensor(data.M, dtype=dtype, device=device)
    Cqd = torch.tensor(data.nle - data.g, dtype=dtype, device=device)
    g_vec = torch.tensor(data.g, dtype=dtype, device=device)
    tau = torch.ones(nv, dtype=dtype, device=device)
    b = tau - Cqd - g_vec

    # Perturbation for stability
    delta_M = torch.randn_like(M) * 1e-6
    M_pert = M + delta_M

    # --- Reference (Gauss-Jordan) ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_ref = gauss_jordan(M, b)
    torch.cuda.synchronize()
    t1 = time.time()
    results['ref_time'].append((t1 - t0) * 1000)

    qddot_ref_pert = gauss_jordan(M_pert, b)
    kappa_ref = (torch.linalg.norm(qddot_ref_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['ref_kappa'].append(kappa_ref.item())

    # --- Neumann Series ---
    torch.cuda.synchronize()
    t0 = time.time()
    neumann_inv = neumann_series_inverse(M)
    qddot_neumann = neumann_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['neumann_time'].append((t1 - t0) * 1000)
    results['neumann_error'].append((torch.linalg.norm(qddot_neumann - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_neumann_pert = neumann_series_inverse(M_pert) @ b
    kappa_neumann = (torch.linalg.norm(qddot_neumann_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                    (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['neumann_kappa'].append(kappa_neumann.item())

    # --- SPAI ---
    torch.cuda.synchronize()
    t0 = time.time()
    spai_inv = spai_inverse(M)
    qddot_spai = spai_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['spai_time'].append((t1 - t0) * 1000)
    results['spai_error'].append((torch.linalg.norm(qddot_spai - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    spai_inv_pert = spai_inverse(M_pert)
    qddot_spai_pert = spai_inv_pert @ b
    kappa_spai = (torch.linalg.norm(qddot_spai_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['spai_kappa'].append(kappa_spai.item())

    # --- HALA ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_hala = hala(M, b, num_neumann=10)
    torch.cuda.synchronize()
    t1 = time.time()
    results['hala_time'].append((t1 - t0) * 1000)
    results['hala_error'].append((torch.linalg.norm(qddot_hala - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_hala_pert = hala(M_pert, b, num_neumann=10)
    kappa_hala = (torch.linalg.norm(qddot_hala_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['hala_kappa'].append(kappa_hala.item())

# ----------------------------
# Summary
# ----------------------------
def summarize(name, times, errors, kappas):
    print(f"{name}: Avg Time (ms): {torch.tensor(times).mean():.2f}, "
          f"Avg Error: {torch.tensor(errors, dtype=torch.float64).mean():.2e}, "
          f"Avg Stability (kappa): {torch.tensor(kappas).mean():.2f}")

# FIXED LINE BELOW
summarize("Gauss-Jordan (ref)", results['ref_time'], [0.0]*num_runs, results['ref_kappa'])
summarize("Neumann Series", results['neumann_time'], results['neumann_error'], results['neumann_kappa'])
summarize("SPAI", results['spai_time'], results['spai_error'], results['spai_kappa'])
summarize("HALA", results['hala_time'], results['hala_error'], results['hala_kappa'])


Using device: cuda
Gauss-Jordan (ref): Avg Time (ms): 0.47, Avg Error: 0.00e+00, Avg Stability (kappa): 28.93
Neumann Series: Avg Time (ms): 0.77, Avg Error: 4.63e-01, Avg Stability (kappa): 346967.22
SPAI: Avg Time (ms): 0.18, Avg Error: 4.03e-01, Avg Stability (kappa): 243378.72
HALA: Avg Time (ms): 0.76, Avg Error: 0.00e+00, Avg Stability (kappa): 28.93


In [ ]:
import pinocchio as pin
import torch
import time

# ----------------------------
# GPU + dtype setup
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64  # match numpy double precision
print("Using device:", device)

# ----------------------------
# Algorithm Implementations
# ----------------------------
def gauss_jordan(M, b):
    return torch.linalg.solve(M, b)

def neumann_series_inverse(M, num_terms=10):
    M0 = torch.diag(torch.diag(M))
    M0_inv = torch.linalg.inv(M0)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - M0_inv @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_terms):
        term = term @ E
        S = S + term
    return S @ M0_inv

def spai_inverse(M):
    return torch.diag(1.0 / torch.diag(M))

def hala(M, b, num_neumann=5):
    G_spai = spai_inverse(M)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - G_spai @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_neumann):
        term = term @ E
        S = S + term
    M_inv_hala = S @ G_spai
    qddot_hala = M_inv_hala @ b
    error = torch.linalg.norm(M @ qddot_hala - b)
    if error > 1e-3:
        qddot_hala = gauss_jordan(M, b)
    return qddot_hala

# ----------------------------
# Setup Pinocchio model
# ----------------------------
urdf_path = '/content/drive/MyDrive/robot_arm_2link.urdf'
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()
nq, nv = model.nq, model.nv

q_lower, q_upper = model.lowerPositionLimit, model.upperPositionLimit
qd_limit = model.velocityLimit

# Ensure finite, reasonable velocity limits
qd_limit_safe = qd_limit.copy()
qd_limit_safe[~torch.isfinite(torch.tensor(qd_limit_safe))] = 1.0
qd_limit_safe = torch.clamp(torch.tensor(qd_limit_safe), 0, 10).numpy()

# ----------------------------
# Random Sampling
# ----------------------------
num_runs = 1000
q_lower_t = torch.tensor(q_lower, dtype=dtype)
q_upper_t = torch.tensor(q_upper, dtype=dtype)
qd_limit_safe_t = torch.tensor(qd_limit_safe, dtype=dtype)

# Uniformly sample between lower and upper joint limits
q_samples = q_lower_t + (q_upper_t - q_lower_t) * torch.rand(num_runs, nq, dtype=dtype)
qd_samples = -qd_limit_safe_t + 2 * qd_limit_safe_t * torch.rand(num_runs, nv, dtype=dtype)

# ----------------------------
# Benchmark Setup
# ----------------------------
results = {
    'ref_time': [], 'neumann_time': [], 'spai_time': [], 'hala_time': [],
    'neumann_error': [], 'spai_error': [], 'hala_error': [],
    'ref_kappa': [], 'neumann_kappa': [], 'spai_kappa': [], 'hala_kappa': []
}

# ----------------------------
# Benchmark Loop
# ----------------------------
for i in range(num_runs):
    q = q_samples[i].cpu().numpy()
    qd = qd_samples[i].cpu().numpy()
    pin.computeAllTerms(model, data, q, qd)

    # Convert Pinocchio outputs to GPU tensors
    M = torch.tensor(data.M, dtype=dtype, device=device)
    Cqd = torch.tensor(data.nle - data.g, dtype=dtype, device=device)
    g_vec = torch.tensor(data.g, dtype=dtype, device=device)
    tau = torch.ones(nv, dtype=dtype, device=device)
    b = tau - Cqd - g_vec

    # Perturbation for stability
    delta_M = torch.randn_like(M) * 1e-6
    M_pert = M + delta_M

    # --- Reference (Gauss-Jordan) ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_ref = gauss_jordan(M, b)
    torch.cuda.synchronize()
    t1 = time.time()
    results['ref_time'].append((t1 - t0) * 1000)

    qddot_ref_pert = gauss_jordan(M_pert, b)
    kappa_ref = (torch.linalg.norm(qddot_ref_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['ref_kappa'].append(kappa_ref.item())

    # --- Neumann Series ---
    torch.cuda.synchronize()
    t0 = time.time()
    neumann_inv = neumann_series_inverse(M)
    qddot_neumann = neumann_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['neumann_time'].append((t1 - t0) * 1000)
    results['neumann_error'].append((torch.linalg.norm(qddot_neumann - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_neumann_pert = neumann_series_inverse(M_pert) @ b
    kappa_neumann = (torch.linalg.norm(qddot_neumann_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                    (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['neumann_kappa'].append(kappa_neumann.item())

    # --- SPAI ---
    torch.cuda.synchronize()
    t0 = time.time()
    spai_inv = spai_inverse(M)
    qddot_spai = spai_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['spai_time'].append((t1 - t0) * 1000)
    results['spai_error'].append((torch.linalg.norm(qddot_spai - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    spai_inv_pert = spai_inverse(M_pert)
    qddot_spai_pert = spai_inv_pert @ b
    kappa_spai = (torch.linalg.norm(qddot_spai_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['spai_kappa'].append(kappa_spai.item())

    # --- HALA ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_hala = hala(M, b, num_neumann=10)
    torch.cuda.synchronize()
    t1 = time.time()
    results['hala_time'].append((t1 - t0) * 1000)
    results['hala_error'].append((torch.linalg.norm(qddot_hala - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_hala_pert = hala(M_pert, b, num_neumann=10)
    kappa_hala = (torch.linalg.norm(qddot_hala_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['hala_kappa'].append(kappa_hala.item())

# ----------------------------
# Summary
# ----------------------------
def summarize(name, times, errors, kappas):
    print(f"{name}: Avg Time (ms): {torch.tensor(times).mean():.2f}, "
          f"Avg Error: {torch.tensor(errors, dtype=torch.float64).mean():.2e}, "
          f"Avg Stability (kappa): {torch.tensor(kappas).mean():.2f}")

# FIXED LINE BELOW
summarize("Gauss-Jordan (ref)", results['ref_time'], [0.0]*num_runs, results['ref_kappa'])
summarize("Neumann Series", results['neumann_time'], results['neumann_error'], results['neumann_kappa'])
summarize("SPAI", results['spai_time'], results['spai_error'], results['spai_kappa'])
summarize("HALA", results['hala_time'], results['hala_error'], results['hala_kappa'])


Using device: cuda
Gauss-Jordan (ref): Avg Time (ms): 0.22, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00
Neumann Series: Avg Time (ms): 0.69, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00
SPAI: Avg Time (ms): 0.15, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00
HALA: Avg Time (ms): 0.60, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00


In [ ]:
# urdf_1000dof_generator.py
N = 100  # Number of DOF

header = '''<?xml version="1.0"?>
<robot name="N_robot">
  <material name="blue">
    <color rgba="0.0 0.0 0.8 1.0"/>
  </material>
  <material name="red">
    <color rgba="0.8 0.0 0.0 1.0"/>
  </material>
  <material name="grey">
    <color rgba="0.7 0.7 0.7 1.0"/>
  </material>
  <material name="dark_grey">
    <color rgba="0.3 0.3 0.3 1.0"/>
  </material>
  <link name="world"/>
  <link name="base_link">
    <visual>
      <geometry>
        <cylinder length="0.05" radius="0.12"/>
      </geometry>
      <material name="dark_grey"/>
    </visual>
    <collision>
      <geometry>
        <cylinder length="0.05" radius="0.12"/>
      </geometry>
    </collision>
    <inertial>
      <mass value="4.0"/>
      <origin rpy="0 0 0" xyz="0.0 0.0 0.0"/>
      <inertia ixx="0.00443333156" ixy="0.0" ixz="0.0" iyy="0.00443333156" iyz="0.0" izz="0.0072"/>
    </inertial>
  </link>
  <joint name="world_joint" type="fixed">
    <parent link="world"/>
    <child link="base_link"/>
    <origin rpy="0.0 0.0 0.0" xyz="0.0 0.0 0.0"/>
  </joint>
'''

link_template = '''
  <link name="link{idx}">
    <visual>
      <geometry>
        <cylinder length="0.18" radius="0.06"/>
      </geometry>
      <material name="blue"/>
    </visual>
    <collision>
      <geometry>
        <cylinder length="0.18" radius="0.06"/>
      </geometry>
    </collision>
    <inertial>
      <mass value="3.7"/>
      <origin rpy="0 0 0" xyz="0.0 0.0 0.0"/>
      <inertia ixx="0.010267495893" ixy="0.0" ixz="0.0" iyy="0.010267495893" iyz="0.0" izz="0.00666"/>
    </inertial>
  </link>
'''

joint_template = '''
  <joint name="joint{idx}" type="revolute">
    <parent link="{parent}"/>
    <child link="link{idx}"/>
    <origin rpy="0.0 0.0 0.0" xyz="0.0 0.0 0.089159"/>
    <axis xyz="0 0 1"/>
    <limit lower="-3.14159" upper="3.14159" effort="150" velocity="3.15"/>
  </joint>
'''

ee = '''
  <link name="ee_link">
    <visual>
      <geometry>
        <box size="0.04 0.04 0.04"/>
      </geometry>
      <material name="grey"/>
    </visual>
    <collision>
      <geometry>
        <box size="0.04 0.04 0.04"/>
      </geometry>
    </collision>
    <inertial>
      <mass value="0.1"/>
      <origin rpy="0 0 0" xyz="0.0 0.0 0.0"/>
      <inertia ixx="0.0001" ixy="0.0" ixz="0.0" iyy="0.0001" iyz="0.0" izz="0.0001"/>
    </inertial>
  </link>
  <joint name="ee_joint" type="fixed">
    <origin rpy="-1.570796325 0 0" xyz="0 0.0823 0"/>
    <parent link="link{last}"/>
    <child link="ee_link"/>
  </joint>
</robot>
'''

# Build the body
body = ""
for i in range(1, N+1):
    body += link_template.format(idx=i)
    parent = "base_link" if i == 1 else f"link{i-1}"
    body += joint_template.format(idx=i, parent=parent)

# Finalize URDF
urdf_str = header + body + ee.format(last=N)

# Save to file
with open("/content/drive/MyDrive/N_robot.urdf", "w") as f:
    f.write(urdf_str)

print("Saved 1000-DOF URDF as 'N_robot.urdf'")


Saved 1000-DOF URDF as 'N_robot.urdf'


In [ ]:
# make sure CUDA is installed
!nvcc --version

# make sure you have a GPU runtime (if this fails go to runtime -> change runtime type)
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Tue Nov 18 20:55:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P0       

In [ ]:
# batched_hala_gpu.py
import pinocchio as pin
import torch
import time
import numpy as np

# ----------------------------
# Config
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
print("Using device:", device)

# Benchmark parameters
num_runs = 1000          # total number of samples
batch_size = 64          # adjust: larger for better GPU utilization
num_neumann = 10         # HALA Neumann terms
neumann_terms_ref = 10   # Neumann for the separate test (same here)
print(f"num_runs={num_runs}, batch_size={batch_size}")

# GPU-safe sync
def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

# ----------------------------
# Algorithm (batched versions)
# ----------------------------
def batched_solve(M_batch, b_batch):
    # M_batch: (B, n, n), b_batch: (B, n)
    # Returns x_batch: (B, n)
    return torch.linalg.solve(M_batch, b_batch.unsqueeze(-1)).squeeze(-1)

def batched_neumann_inverse(M_batch, num_terms=5):
    # M_batch: (B, n, n)
    B, n, _ = M_batch.shape
    I = torch.eye(n, dtype=M_batch.dtype, device=M_batch.device).unsqueeze(0).expand(B, -1, -1)
    # Use diagonal preconditioner M0 = diag(diag(M)), compute M0_inv batchwise
    diag = torch.diagonal(M_batch, dim1=1, dim2=2)  # (B, n)
    eps = 1e-20
    M0_inv = torch.zeros_like(M_batch)
    inv_diag = 1.0 / (diag + eps)
    M0_inv = torch.diag_embed(inv_diag)  # (B, n, n)
    R = I - M0_inv @ M_batch  # (B, n, n)
    acc = M0_inv.clone()
    term = torch.eye(n, dtype=M_batch.dtype, device=M_batch.device).unsqueeze(0).expand(B, -1, -1)
    for _ in range(1, num_terms):
        term = term @ R
        acc = acc + term @ M0_inv
    return acc  # approximate inverse batch (B,n,n)

def batched_spai_inverse(M_batch):
    # simple SPAI: only diag inverse (cheap)
    diag = torch.diagonal(M_batch, dim1=1, dim2=2)
    inv_diag = 1.0 / (diag + 1e-20)
    return torch.diag_embed(inv_diag)

def batched_hala_solve(M_batch, b_batch, num_neumann=5):
    # returns x_batch (B,n)
    G_spai = batched_spai_inverse(M_batch)               # (B,n,n)
    I = torch.eye(M_batch.shape[1], dtype=M_batch.dtype, device=M_batch.device).unsqueeze(0).expand(M_batch.shape[0], -1, -1)
    E = I - G_spai @ M_batch
    S = I.clone()
    term = I.clone()
    for _ in range(1, num_neumann):
        term = term @ E
        S = S + term
    M_inv_hala = S @ G_spai
    x = (M_inv_hala @ b_batch.unsqueeze(-1)).squeeze(-1)
    # fallback to exact if residual large (vectorized check)
    residual = torch.linalg.norm(M_batch @ x.unsqueeze(-1) - b_batch.unsqueeze(-1), dim=(1,2))
    mask = residual > 1e-3
    if mask.any():
        # compute exact for those indices
        if mask.all():
            x_exact = batched_solve(M_batch, b_batch)
            x = x_exact
        else:
            # compute exact only for masked entries
            M_mask = M_batch[mask]
            b_mask = b_batch[mask]
            x_mask = batched_solve(M_mask, b_mask)
            x[mask] = x_mask
    return x

# ----------------------------
# Load Pinocchio model (CPU)
# ----------------------------
urdf_path = '/content/drive/MyDrive/N_robot.urdf'  # change to your path
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()
nq, nv = model.nq, model.nv
print("nq, nv:", nq, nv)

q_lower, q_upper = model.lowerPositionLimit, model.upperPositionLimit
qd_limit = model.velocityLimit

# sanitize qd limits
qd_limit_safe = qd_limit.copy()
qd_limit_safe[~np.isfinite(qd_limit_safe)] = 1.0
qd_limit_safe = np.clip(qd_limit_safe, 0, 10)

# prepare random samples on CPU (numpy)
rng = np.random.default_rng(12345)
q_samples = rng.uniform(low=q_lower, high=q_upper, size=(num_runs, nq))
qd_samples = rng.uniform(low=-qd_limit_safe, high=qd_limit_safe, size=(num_runs, nv))

# ----------------------------
# Helper: process batches -> collect M and b on CPU then copy to GPU once
# ----------------------------
def collect_batch_matrices(start_idx, end_idx):
    """Compute Pinocchio dynamics for indices [start_idx, end_idx) and return
       stacked numpy arrays M_batch_cpu (B,n,n) and b_batch_cpu (B,n).
    """
    B = end_idx - start_idx
    Ms = np.empty((B, nv, nv), dtype=np.float64)
    bs = np.empty((B, nv), dtype=np.float64)
    for i, idx in enumerate(range(start_idx, end_idx)):
        q = q_samples[idx]
        qd = qd_samples[idx]
        pin.computeAllTerms(model, data, q, qd)
        M_cpu = data.M.copy()           # shape (nv,nv)
        Cqd_cpu = (data.nle - data.g).copy()
        g_cpu = data.g.copy()
        tau_cpu = np.ones(nv, dtype=np.float64)
        b_cpu = tau_cpu - Cqd_cpu - g_cpu
        Ms[i] = M_cpu
        bs[i] = b_cpu
    return Ms, bs

# ----------------------------
# Run batched experiment
# ----------------------------
results = {
    'ref_time': [], 'neumann_time': [], 'spai_time': [], 'hala_time': [],
    'ref_err': [], 'neumann_err': [], 'spai_err': [], 'hala_err': [],
    'ref_kappa': [], 'neumann_kappa': [], 'spai_kappa': [], 'hala_kappa': []
}

num_batches = (num_runs + batch_size - 1) // batch_size

for bidx in range(num_batches):
    s = bidx * batch_size
    e = min(s + batch_size, num_runs)
    B = e - s
    # 1) collect on CPU
    Ms_cpu, bs_cpu = collect_batch_matrices(s, e)           # shapes (B,nv,nv), (B,nv)

    # 2) single copy to GPU
    M_batch = torch.from_numpy(Ms_cpu).to(device=device, dtype=dtype)   # (B,n,n)
    b_batch = torch.from_numpy(bs_cpu).to(device=device, dtype=dtype)   # (B,n)

    # small perturbations on GPU
    delta_M = (torch.randn_like(M_batch) * 1e-6).to(device=device)
    M_batch_pert = M_batch + delta_M

    # --- Reference batched solve ---
    sync()
    t0 = time.time()
    x_ref = batched_solve(M_batch, b_batch)   # (B,n)
    sync()
    t1 = time.time()
    results['ref_time'].append((t1 - t0)*1000.0)

    # compute kappa_ref per-batch entry
    x_ref_pert = batched_solve(M_batch_pert, b_batch)
    num = torch.linalg.norm(x_ref_pert - x_ref, dim=1)   # (B,)
    den = torch.linalg.norm(x_ref, dim=1) + 1e-20
    kappa_ref_batch = (num / den) / (torch.linalg.norm(delta_M.reshape(B, -1), dim=1) / (torch.linalg.norm(M_batch.reshape(B, -1), dim=1) + 1e-20))
    results['ref_kappa'].extend(kappa_ref_batch.cpu().tolist())

    # --- Neumann (batch) ---
    sync()
    t0 = time.time()
    inv_neu = batched_neumann_inverse(M_batch, num_terms=num_neumann)   # (B,n,n)
    x_neu = (inv_neu @ b_batch.unsqueeze(-1)).squeeze(-1)
    sync()
    t1 = time.time()
    results['neumann_time'].append((t1 - t0)*1000.0)
    results['neumann_err'].extend((torch.linalg.norm(x_neu - x_ref, dim=1) / (torch.linalg.norm(x_ref, dim=1) + 1e-20)).cpu().tolist())

    x_neu_pert = (batched_neumann_inverse(M_batch_pert, num_terms=num_neumann) @ b_batch.unsqueeze(-1)).squeeze(-1)
    kappa_neu_batch = (torch.linalg.norm(x_neu_pert - x_ref, dim=1) / (torch.linalg.norm(x_ref, dim=1) + 1e-20)) / \
                      (torch.linalg.norm(delta_M.reshape(B, -1), dim=1) / (torch.linalg.norm(M_batch.reshape(B, -1), dim=1) + 1e-20))
    results['neumann_kappa'].extend(kappa_neu_batch.cpu().tolist())

    # --- SPAI (batch) ---
    sync()
    t0 = time.time()
    inv_spai = batched_spai_inverse(M_batch)
    x_spai = (inv_spai @ b_batch.unsqueeze(-1)).squeeze(-1)
    sync()
    t1 = time.time()
    results['spai_time'].append((t1 - t0)*1000.0)
    results['spai_err'].extend((torch.linalg.norm(x_spai - x_ref, dim=1) / (torch.linalg.norm(x_ref, dim=1) + 1e-20)).cpu().tolist())

    x_spai_pert = (batched_spai_inverse(M_batch_pert) @ b_batch.unsqueeze(-1)).squeeze(-1)
    kappa_spai_batch = (torch.linalg.norm(x_spai_pert - x_ref, dim=1) / (torch.linalg.norm(x_ref, dim=1) + 1e-20)) / \
                       (torch.linalg.norm(delta_M.reshape(B, -1), dim=1) / (torch.linalg.norm(M_batch.reshape(B, -1), dim=1) + 1e-20))
    results['spai_kappa'].extend(kappa_spai_batch.cpu().tolist())

    # --- HALA (batch) ---
    sync()
    t0 = time.time()
    x_hala = batched_hala_solve(M_batch, b_batch, num_neumann=num_neumann)
    sync()
    t1 = time.time()
    results['hala_time'].append((t1 - t0)*1000.0)
    results['hala_err'].extend((torch.linalg.norm(x_hala - x_ref, dim=1) / (torch.linalg.norm(x_ref, dim=1) + 1e-20)).cpu().tolist())

    x_hala_pert = batched_hala_solve(M_batch_pert, b_batch, num_neumann=num_neumann)
    kappa_hala_batch = (torch.linalg.norm(x_hala_pert - x_ref, dim=1) / (torch.linalg.norm(x_ref, dim=1) + 1e-20)) / \
                       (torch.linalg.norm(delta_M.reshape(B, -1), dim=1) / (torch.linalg.norm(M_batch.reshape(B, -1), dim=1) + 1e-20))
    results['hala_kappa'].extend(kappa_hala_batch.cpu().tolist())

# ----------------------------
# Summaries (aggregate across batches)
# ----------------------------
def summarize_list(name, times, errors, kappas):
    times = np.array(times, dtype=np.float64)
    errors = np.array(errors, dtype=np.float64)
    kappas = np.array(kappas, dtype=np.float64)
    print(f"{name}: Avg Time (ms): {times.mean():.3f}, Avg Error: {errors.mean():.3e}, Avg Kappa: {kappas.mean():.3f}")

summarize_list("Gauss-Jordan (ref)", results['ref_time'], [0.0]*num_runs, results['ref_kappa'])
summarize_list("Neumann Series", results['neumann_time'], results['neumann_err'], results['neumann_kappa'])
summarize_list("SPAI", results['spai_time'], results['spai_err'], results['spai_kappa'])
summarize_list("HALA", results['hala_time'], results['hala_err'], results['hala_kappa'])

# Optionally print a quick performance comparison CPU vs GPU:
# Note: We measured only GPU batched times above. To compare, measure an equivalent batched CPU run.


Using device: cuda
num_runs=1000, batch_size=64
nq, nv: 100 100
Gauss-Jordan (ref): Avg Time (ms): 24.197, Avg Error: 0.000e+00, Avg Kappa: 1008.255
Neumann Series: Avg Time (ms): 29.212, Avg Error: 9.579e+15, Avg Kappa: 2636808417595906064384.000
SPAI: Avg Time (ms): 0.237, Avg Error: 8.585e-01, Avg Kappa: 236314.146
HALA: Avg Time (ms): 16.133, Avg Error: 0.000e+00, Avg Kappa: 1008.255
